# Chain of thought prompting

In [1]:
import json
import re
import anthropic
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
ANTHRPOIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
client = anthropic.Anthropic(api_key = ANTHRPOIC_API_KEY)

In [4]:
def _parse_json(text: str) -> dict:
    text = text.strip()
    m = re.search(r'```(?:json)?\s*\n?([\s\S]+?)\n?\s*```', text)
    if m:
        text = m.group(1).strip()
    start, end = text.find('{'), text.find('}')+1
    if start != -1 and end > start:
        text = text[start:end]
    return json.loads(text)

In [ ]:
CLASSIFY_SYSTEM = """
You are a support ticket classifier for Meridian cloud.

Classify each ticket into exactly one of these categories:
    - billing_dispute: customer contests a charge or invoice total
    - billing_inquiry: customer asks about charges without disputing them
    - payment_failure: a payment attempt was declined or could not be processed
    - technical_error: a product or API is not working as expected
    - account_access: login, password, or permission issues
    - feature_request: customer asks for capability that doesn't exist yet
    - other: doesn't fit the above categories

Return a JSON object with fields "category" and "confidence" (high/medium/low).

Examples:

Input: "I was charged $120 but my plan is $80/month. Please explain.
Output: {"category": "billing_dispute", "confidence": "high"}

Input: "When does my billing cycle reset every month?"
Output: {"category": "billing_inquiry", "confidence": "high"}

Input: "My card ending with 1234 was declined when I tried to upgrade."
Output: {"category": "payment_failure", "confidence": "high"}

Input: "The /v2/events endpoint has been returning 503 error since morning."
Output: {"category": "technical_error", "confidence": "high"}

Input: "I need to add a team member but the invite button is disabled."
Output: {"category": "account_access", "confidence": "medium"}

Return only a JSON object.
"""